In [3]:
import networkx as nx
import pandas as pd

# Load STRING network
string_data = pd.read_csv("~/Documents/GitHub/drugrep_tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)
string_data

,gene1,gene2,weight
0,ARF5,DYRK4,0.166000
1,ARF5,PPP5C,0.254968
2,ARF5,MAP4K5,0.157276
3,ARF5,RALBP1,0.156000
4,ARF5,PKP2,0.160210
...,...,...,...
1972242,ZNF788,ANKRD65,0.162811
1972243,ZNF788,PCSK5,0.215378
1972244,ZNF788,OBSCN,0.236000
1972245,ZNF788,RAB5C,0.195550


In [4]:
# Create graph (assuming 'gene1', 'gene2', 'weight' columns)
G = nx.Graph()
for _, row in string_data.iterrows():
    G.add_edge(row['gene1'], row['gene2'], weight=1-row['weight'])


In [3]:
"ARF5" in G

True

In [5]:
node_types = pd.read_csv("~/Documents/GitHub/drugrep_tb/results/uniformly_processed/NODE_TYPES.txt",sep="\t",names=['#Node','type'],header=0)
# Assuming the 'type' column contains labels like 'node' and 'target'
sources = node_types[node_types['type'] == 'source']['#Node'].tolist()
targets = node_types[node_types['type'] == 'target']['#Node'].tolist()

shortest_paths = {}
# Compute shortest paths, skipping nodes not in the graph
for source in sources:
    if source not in G:
        print(f"Source {source} is not in the graph. Skipping...")
        continue
    for target in targets:
        if target not in G:
            print(f"Target {target} is not in the graph. Skipping...")
            continue
        try:
            # Compute shortest path if it exists
            if nx.has_path(G, source, target):
                path = nx.shortest_path(G, source=source, target=target, weight='weight')
                shortest_paths[(source, target)] = path
            else:
                shortest_paths[(source, target)] = None
        except Exception as e:
            print(f"Error processing source-target pair ({source}, {target}): {e}")
            shortest_paths[(source, target)] = None


Target CCN2 is not in the graph. Skipping...
Target POLR1HASP is not in the graph. Skipping...
Target ACKR1 is not in the graph. Skipping...
Target RARRES1 is not in the graph. Skipping...
Target BTF3P11 is not in the graph. Skipping...
Target LRMDA is not in the graph. Skipping...
Target COP 1 is not in the graph. Skipping...
Target ZNF542P is not in the graph. Skipping...
Target AOC1 is not in the graph. Skipping...
Target ERVK2 is not in the graph. Skipping...
Target LINC00473 is not in the graph. Skipping...
Target NKAIN3 is not in the graph. Skipping...
Target AFG2A is not in the graph. Skipping...
Target H2AC25 is not in the graph. Skipping...
Target HPSE is not in the graph. Skipping...
Target CCN2 is not in the graph. Skipping...
Target POLR1HASP is not in the graph. Skipping...
Target ACKR1 is not in the graph. Skipping...
Target RARRES1 is not in the graph. Skipping...
Target BTF3P11 is not in the graph. Skipping...
Target LRMDA is not in the graph. Skipping...
Target COP 1 i

In [6]:
shortest_paths

{('CXCL2', 'MAPK4'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'HSP90AA1',
  'RAF1',
  'MAP2K1',
  'MAPK1',
  'MAPKAPK5',
  'MAPK4'],
 ('CXCL2', 'VIP'): ['CXCL2', 'DPP4', 'VIP'],
 ('CXCL2', 'G6PD'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'G6PD'],
 ('CXCL2', 'HLA-DQB1'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'HLA-DQA1',
  'HLA-DQB1'],
 ('CXCL2', 'BHLHE40'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'BHLHE40'],
 ('CXCL2', 'CSF1'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'CBL',
  'CSF1R',
  'CSF1'],
 ('CXCL2', 'JUND'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'JUN',
  'FOS',
  'JUND'],
 ('CXCL2', 'HSPA1A'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'DNAJA2',
  'HSPA1A'],
 ('CXCL2', 'ATF3'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'JUN', 'ATF3'],
 ('CXCL2', 'PAX2'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'HDAC1', 'RB1', 'PAX2'],
 ('CXCL2', 'HTR4'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'ADRB2', 'HTR4'],
 ('CXCL2', 'GLP1R'): ['CXCL2', 'CXC

In [7]:
from collections import Counter

# Flatten the shortest paths
path_nodes = [node for path in shortest_paths.values() if path for node in path]
node_participation = Counter(path_nodes)

In [8]:
# Create a subgraph based on relevant shortest paths
relevant_edges = [(u, v) for path in shortest_paths.values() if path for u, v in zip(path, path[1:])]
subgraph = G.edge_subgraph(relevant_edges)

# Compute betweenness centrality
bc_scores = nx.betweenness_centrality(subgraph, weight='weight')

In [9]:
sorted_bc = sorted(bc_scores.items(), key=lambda x: x[1], reverse=True)

In [10]:
# Define BC cutoff (example: top 2.5%)
top_nodes_cutoff = int(len(sorted_bc) * 0.025)
important_nodes = [node for node, bc in sorted_bc[:top_nodes_cutoff]]

In [11]:
len(important_nodes)
important_nodes

['UBC',
 'CBL',
 'HSP90AA1',
 'RPS3',
 'ATP5A1',
 'ATP5B',
 'EGFR',
 'ESR1',
 'HSPA8',
 'BCAS2',
 'TP53',
 'CCNB1']

In [12]:
important_nodes_ = set(important_nodes) - set(sources) - set(targets)
print(type(shortest_paths))
# Subset dictionary
filtered_paths = {
    key: path for key, path in shortest_paths.items()
    if any(gene in path for gene in important_nodes_)
}

<class 'dict'>


In [13]:
print(len(shortest_paths))
print(len(filtered_paths))
filtered_paths

1414
1373


{('CXCL2', 'MAPK4'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'HSP90AA1',
  'RAF1',
  'MAP2K1',
  'MAPK1',
  'MAPKAPK5',
  'MAPK4'],
 ('CXCL2', 'G6PD'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'G6PD'],
 ('CXCL2', 'HLA-DQB1'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'HLA-DQA1',
  'HLA-DQB1'],
 ('CXCL2', 'BHLHE40'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'BHLHE40'],
 ('CXCL2', 'CSF1'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'CBL',
  'CSF1R',
  'CSF1'],
 ('CXCL2', 'JUND'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'JUN',
  'FOS',
  'JUND'],
 ('CXCL2', 'HSPA1A'): ['CXCL2',
  'CXCR2',
  'IL8',
  'RELA',
  'UBC',
  'DNAJA2',
  'HSPA1A'],
 ('CXCL2', 'ATF3'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'JUN', 'ATF3'],
 ('CXCL2', 'HTR4'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'ADRB2', 'HTR4'],
 ('CXCL2', 'GLP1R'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'GLP1R'],
 ('CXCL2', 'TSC2'): ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'TSC2'],
 ('CXCL2', 'HERC2

In [14]:
# Convert to DataFrame
# Convert to DataFrame
filtered_paths_df = pd.DataFrame([
    {"Source": key[0], "Target": key[1], "Path": ",".join(path)}
    for key, path in filtered_paths.items()
])


filtered_paths_df
# Write DataFrame to CSV
filtered_paths_df.to_csv("~/Documents/GitHub/drugrep_tb/results/uniformly_processed/filtered_shortest_paths_bcscores.csv", index=False,sep='\t')

In [15]:
# Save results
pd.DataFrame(sorted_bc, columns=['Node', 'BC_Score']).to_csv("~/Documents/GitHub/drugrep_tb/results/uniformly_processed/bc_scores.csv", index=False)

In [16]:
import pandas as pd

# Load STRING network
# Assuming STRING network has 'gene1', 'gene2', and 'weight' columns
string_data = pd.read_csv("~/Documents/GitHub/drugrep_tb/data/metadata/STRING_hs_interactions.txt", sep="\t", names=['gene1','gene2','weight'],header=0)

# Extract all unique genes from the shortest paths
genes_in_paths = set(
    gene for path in filtered_paths.values() if path for gene in path
)

# Subset the STRING network by filtering rows where both genes are in `genes_in_paths`
subset_network = string_data[
    (string_data['gene1'].isin(genes_in_paths)) &
    (string_data['gene2'].isin(genes_in_paths))
]

# Save the subsetted network to a CSV
subset_network.to_csv("~/Documents/GitHub/drugrep_tb/results/uniformly_processed/subset_string_network_shortestpaths_bcscores.csv", index=False)

# Print a summary
print(f"Original STRING network size: {string_data.shape[0]} edges")
print(f"Subset STRING network size: {subset_network.shape[0]} edges")


Original STRING network size: 1972247 edges
Subset STRING network size: 11192 edges


In [17]:
# Compute average BC scores for each path
path_avg_bc = {}
for key, path in shortest_paths.items():
    if path:  # Ensure the path is not None
        avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
        path_avg_bc[key] = avg_bc

# Group paths by source node
paths_by_source = {}
for (source, target), avg_bc in path_avg_bc.items():
    if source not in paths_by_source:
        paths_by_source[source] = []
    paths_by_source[source].append(((source, target), avg_bc))

# Get top paths for each source node
top_percentage = 0.02  # Adjust this as needed
top_paths_per_source = {}

for source, paths in paths_by_source.items():
    # Sort paths by average BC score for the current source
    sorted_paths = sorted(paths, key=lambda x: x[1], reverse=True)
    num_top_paths = max(1, int(len(sorted_paths) * top_percentage))  # At least 1 path
    top_paths = sorted_paths[:num_top_paths]

    # Filter paths to include only those passing through important nodes
    filtered_paths = {
        key: shortest_paths[key]
        for key, avg_bc in top_paths
        if any(gene in shortest_paths[key] for gene in important_nodes_)
    }

    top_paths_per_source[source] = filtered_paths

# Print results for each source node
for source, paths in top_paths_per_source.items():
    print(f"Source: {source}")
    print(f"Number of top paths: {len(paths)}")
    for key, path in paths.items():
        print(f"  Target: {key[1]}, Path: {path}")
    print("-" * 30)



Source: CXCL2
Number of top paths: 4
  Target: CCNE1, Path: ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'CCNE1']
  Target: TSC2, Path: ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'TSC2']
  Target: NFE2L2, Path: ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'NFE2L2']
  Target: GRB2, Path: ['CXCL2', 'CXCR2', 'IL8', 'RELA', 'UBC', 'CBL', 'GRB2']
------------------------------
Source: HMGCR
Number of top paths: 4
  Target: HIF1A, Path: ['HMGCR', 'UBC', 'HIF1A']
  Target: CCNE1, Path: ['HMGCR', 'UBC', 'CCNE1']
  Target: TSC2, Path: ['HMGCR', 'UBC', 'TSC2']
  Target: NFE2L2, Path: ['HMGCR', 'UBC', 'NFE2L2']
------------------------------
Source: MMP1
Number of top paths: 4
  Target: HIF1A, Path: ['MMP1', 'TIMP1', 'UBC', 'HIF1A']
  Target: CCNE1, Path: ['MMP1', 'TIMP1', 'UBC', 'CCNE1']
  Target: TSC2, Path: ['MMP1', 'TIMP1', 'UBC', 'TSC2']
  Target: NFE2L2, Path: ['MMP1', 'TIMP1', 'UBC', 'NFE2L2']
------------------------------
Source: PTGS2
Number of top paths: 4
  Target: TSC2, Path: ['PTGS2', 'TP53',

In [18]:
# Prepare data for DataFrame
csv_rows = []
for source, paths in top_paths_per_source.items():
    for (source_node, target_node), path in paths.items():
        if path:  # Ensure the path is not None
            # Calculate average BC score
            avg_bc = sum(bc_scores.get(node, 0) for node in path) / len(path)
            # Add row to list
            csv_rows.append({
                "Source": source_node,
                "Target": target_node,
                "Path": " -> ".join(path),  # Convert path list to a string
                "Average_BC_Score": avg_bc
            })

# Convert to DataFrame
filtered_paths_df = pd.DataFrame(csv_rows)

# Write DataFrame to CSV
output_file = "~/Documents/GitHub/drugrep_tb/results/uniformly_processed/filtered_shortest_paths_bcscores_persource.csv"
filtered_paths_df.to_csv(output_file, index=False, sep='\t')

# Print confirmation
print(f"Top paths per source written to: {output_file}")


Top paths per source written to: ~/Documents/GitHub/drugrep_tb/results/uniformly_processed/filtered_shortest_paths_bcscores_persource.csv


In [19]:
# Extract all unique genes (nodes) between sources and targets
intermediate_genes = set()

for source, paths in top_paths_per_source.items():
    for key, path in paths.items():
        if path:  # Ensure the path is not None
            # Exclude source and target nodes
            intermediate_genes.update(path[1:-1])

print(f"Number of unique intermediate genes: {len(intermediate_genes)}")
print("Genes between sources and targets:")
print(intermediate_genes)


Number of unique intermediate genes: 10
Genes between sources and targets:
{'MYD88', 'RELA', 'CXCR2', 'UBC', 'GRB2', 'CBL', 'TIMP1', 'TRAF6', 'IL8', 'TP53'}


In [ ]:
# Extract all unique genes from the top paths per source
genes_in_paths = set(
    gene
    for paths in top_paths_per_source.values()  # Iterate over sources
    for path in paths.values()  # Iterate over paths for each source
    if path  # Ensure the path is not None
    for gene in path  # Extract all genes in the path
)

subnetwork_genes = genes_in_paths.union(set(sources)).union(set(targets)) 

# Subset the STRING network by filtering rows where both genes are in `genes_in_paths`
subset_network = string_data[
    (string_data['gene1'].isin(subnetwork_genes)) &
    (string_data['gene2'].isin(subnetwork_genes))
]

# Save the subsetted network to a CSV
subset_network.to_csv("~/Documents/GitHub/drugrep_tb/results/uniformly_processed/subset_string_network_shortestpaths_bcscores_top2pctpersource.csv", index=False)

# Print a summary
print(f"Original STRING network size: {string_data.shape[0]} edges")
print(f"Subset STRING network size: {subset_network.shape[0]} edges")



TypeError: unsupported operand type(s) for +: 'set' and 'set'